# 01 — Geração do Dataset Simulado

Este notebook gera um dataset simulado de controle de qualidade farmacêutico,
baseado em parâmetros realistas de um processo de compressão de comprimidos.

**Parâmetros simulados:**
- Variáveis de processo: temperatura, umidade, pressão de compressão
- Atributos críticos de qualidade (CQAs): dureza, friabilidade, peso médio, dissolução
- Variável alvo: status do lote (APROVADO / REPROVADO)

In [5]:
import numpy as np
import pandas as pd
import os

np.random.seed(42)
N_LOTES = 500

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# --- Variáveis de processo ---
temp_processo     = np.random.normal(loc=25.0, scale=1.5, size=N_LOTES)       # °C
umidade_relativa  = np.random.normal(loc=45.0, scale=5.0, size=N_LOTES)       # %
pressao_compressao = np.random.normal(loc=12.0, scale=1.2, size=N_LOTES)      # kN

# --- CQAs com correlação aos parâmetros de processo ---
dureza_media   = 80 + 2.5 * pressao_compressao + np.random.normal(0, 3, N_LOTES)   # N
friabilidade   = np.clip(0.5 - 0.05 * pressao_compressao + 0.02 * umidade_relativa
                         + np.random.normal(0, 0.1, N_LOTES), 0.1, 3.0)             # %
peso_medio     = np.random.normal(loc=500, scale=5, size=N_LOTES)                   # mg
dissolucao     = np.clip(115.5 - 0.3 * temp_processo - 0.4 * umidade_relativa
                         + np.random.normal(0, 2, N_LOTES), 60, 100)                # %

# --- Metadados ---
datas = pd.date_range(start='2023-01-01', periods=N_LOTES, freq='12h')
turnos = np.random.choice(['A', 'B', 'C'], size=N_LOTES)
lotes  = [f'LOT-{str(i+1).zfill(4)}' for i in range(N_LOTES)]

In [ ]:
# --- Critérios de reprovação (baseados em especificações típicas) ---
# Dureza: 70–120 N | Friabilidade: < 1.0% | Dissolução: >= 80% | Peso: 475–525 mg
reprovado = (
    (dureza_media < 70) | (dureza_media > 120) |
    (friabilidade >= 1.0) |
    (dissolucao < 80) |
    (peso_medio < 475) | (peso_medio > 525)
)

status_lote = np.where(reprovado, 'REPROVADO', 'APROVADO')

print(f"Total de lotes: {N_LOTES}")
print(f"Aprovados:  {(status_lote == 'APROVADO').sum()} ({(status_lote == 'APROVADO').mean():.1%})")
print(f"Reprovados: {(status_lote == 'REPROVADO').sum()} ({(status_lote == 'REPROVADO').mean():.1%})")

In [ ]:
# --- Montar DataFrame ---
df = pd.DataFrame({
    'lote':               lotes,
    'data_producao':      datas,
    'turno':              turnos,
    'temp_processo':      temp_processo.round(2),
    'umidade_relativa':   umidade_relativa.round(2),
    'pressao_compressao': pressao_compressao.round(2),
    'dureza_media':       dureza_media.round(2),
    'friabilidade':       friabilidade.round(3),
    'peso_medio':         peso_medio.round(2),
    'dissolucao':         dissolucao.round(2),
    'status_lote':        status_lote
})

df.head(10)

In [ ]:
# --- Salvar dataset bruto ---
os.makedirs('../data/raw', exist_ok=True)
df.to_csv('../data/raw/lotes_qc.csv', index=False)
print("Dataset salvo em: data/raw/lotes_qc.csv")
print(f"Shape: {df.shape}")
df.info()